# Intro

A collection of definitions and functions (adapted mostly from Joan Solà (2017) and Adamy (2025) for the observability aspect)



In general, we have three vectors for our state:

- true state
- nominal state
- error state

The true state is what we believe reality looks like, neither our measurements nor our predicted values will never be axactly the truth. This state is therefore never really used in the application but it is helpful to keep as a defintion for our equation simplification later.

The nominal state is the filters best guess at what we predict the true state to be like. It is only an approxamation but with a descent filter implementation, the nominal state is quite close to the true state.

The error state is a little harder to grasp. It essentially describes the error as a value of difference between the nominal state, the true state but most importantly the measurable values. It often times includes uncertainty coming from noise during measurement and also includes the taken measurement and defines how far away the predicted value is from the measured one. We use the error state later on to ingest the accumulated error into our nominal state.

## Preliminaries

\begin{align*}
q\{\phi \} &\triangleq \text{Exp}(\phi) = \begin{bmatrix} \cos(\frac{||\phi||}{2}) \\ \frac{\phi}{||\phi||} \sin(\frac{||\phi||}{2}) \end{bmatrix} \approx \begin{bmatrix} 1 \\ \frac{\phi}{2} \end{bmatrix} \text{ for small angles.} \\
\\
[\omega]_\times &\triangleq \begin{bmatrix}
0 & -\omega_z & \omega_y \\
\omega_z & 0 & -\omega_x \\
-\omega_y & \omega_x & 0
\end{bmatrix}
\end{align*}
The $[\cdot]_\times$ operator establishes an isomorphism mapping a 3D vector $\omega \in \mathbb{R}^3$ to its Lie algebra matrix representation $[\omega]_\times \in \mathfrak{so}(3)$ (See section 1.4.3 in Solà). In practice, this is useful for the error state kinematics for the velocity error in the:
$$
\delta v \leftarrow \delta v + (\underbrace{-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta} - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n}
$$
What is actually happening is that we want to take the acceleration from the accelerometer ($a -a_b$) together with the current orientation ($\mathbf{R}^{\{q\}}$) and some additional error ($\delta \theta$). But since multiplying by the skew symmetric matrix is like taking the cross product, we can use the property $a \times b = -b \times a$ and rewrite it to
$$
\mathbf{R}^{\{q\}} [\delta \theta]_\times (\tilde{a} - a_b) \qquad \Rightarrow \qquad -\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta
$$

---
# Input

The IMU is equipped with an accelerometer and gyroscope. They measure the acceleration and the angular velocity respectively of the IMU. They provide the inputs $u$ as followed:


\begin{align*}
&\text{Accelerometer input: } &\tilde a && &\text{noise: } &\tilde a_n \\
&\text{Gyroscope input: } &\tilde \omega && &\text{noise: } &\tilde \omega_n\\
\end{align*}

We assume these inputs come with a certain amount of gaussian distributed noise which we will have to account for in calculations. All of the input and the noise is defined locally.

---
# Nominal State

## IMU

Our state consists of a vector of length 16 which represents the predicted position and orientation of the IMU (The bias terms have additional noise parameters $a_{bn}$ and $\omega_{bn}$).

\begin{align*}
x_\text{IMU} = \begin{bmatrix} p & v & q & a_b & \omega_b \end{bmatrix}^\top \qquad \qquad x_\text{IMU} \in SO(3) \times \left(\mathbb{R}^3\right)^4
\end{align*}

### Kinematics

\begin{align*}
\dot p &= v & \dot a_b &= 0 \\
\text{Continuous Case} \qquad \qquad \dot v &= \mathbf{R}^{\{q\}}(\tilde a - a_b) + g & \dot \omega_b &= 0 \\
\dot q &= \frac{1}{2} q \otimes (\tilde \omega - \omega_b) \\
\\
p &\leftarrow p + v\Delta t + \frac{1}{2}( \mathbf{R}^{\{q\}} (\tilde a - a_b) + g)\Delta t^2 \qquad & a_b &\leftarrow a_b \\
\text{Discrete Case} \qquad \qquad v &\leftarrow v + (\mathbf{R}^{\{q\}}(\tilde a - a_b) + g)\Delta t & \omega_b &\leftarrow \omega_b \\
q &\leftarrow q \otimes q\{(\tilde \omega - \omega_b) \Delta t\} \\
\end{align*}

## Speakers

Each speaker is represented as their position, velocity and orientation

\begin{align*}
x_s = \begin{bmatrix} p_i & v_i & q_i \end{bmatrix}^\top \qquad \qquad x_s \in \prod_{i = 1}^n \left(SO(3) \times \left(\mathbb{R}^3\right)^2 \right)
\end{align*}

### Kinematics

\begin{align*}
\dot p_i &= v_i \\
\text{Continuous Case} \qquad \qquad \dot v_i &= 0 \\
\dot q_i &= 0 \\
\\
p_i &\leftarrow p_i + v_i \Delta t \\
\text{Discrete Case} \qquad \qquad v_i &\leftarrow v_i \\
q_i &\leftarrow q_i \\
\end{align*}

## Combined State

\begin{align*}
x = \begin{bmatrix} p & v & q & a_b & \omega_b & p_i & v_i & q_i & p_{i + 1} & v_{i + 1} & q_{i + 1} & \cdots \end{bmatrix}^\top
\end{align*}

To calculate observability for the system, I will only use one speaker and assume it holds for the case of multiple speakers. This will be calculated later.

---
# Error State

## IMU

The error state is a vector of length 15 which represents the accumulated error of the IMU.

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b \end{bmatrix}^\top \qquad \qquad x \in \left(\mathbb{R}^3\right)^5
\end{align*}

We are expressing the quaternion error as a vector with three elements since a quaternion has 3 DoF and therefore only needs this many parameters to be corrected.

### Kinematics

\begin{align*}
\dot{\delta p} &= \delta v & \dot{\delta a_b} &= a_{bn} \\
\text{Continuous Case} \qquad \qquad \dot{\delta v} &= -\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b - \mathbf{R}^{\{q\}} \tilde a_n & \dot{\delta \omega_b} &= \omega_{bn} \\
\dot{\delta \theta} &= -[\tilde \omega - \omega_b]_\times \delta \theta - \delta \omega_b - \tilde \omega_n \\
\\
\delta p &\leftarrow \delta p + \delta v\Delta t & \delta a_b &\leftarrow \delta a_b + \delta a_n \\
\text{Discrete Case} \qquad \qquad \delta v &\leftarrow \delta v + (-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n} \qquad & \delta \omega_b &\leftarrow \delta \omega_b + \delta \omega_n \\
\delta \theta &\leftarrow (\mathbf{R}^{\{(\tilde \omega - \omega_b) \Delta t\}})^\top \delta \theta - \delta \omega_b \Delta t + \delta \theta_{n}
\end{align*}

## Speakers

\begin{align*}
\delta x_s = \begin{bmatrix} \delta p_i & \delta v_i & \delta \theta_i \end{bmatrix}^\top \qquad \qquad x_s \in \left(\mathbb{R}^3\right)^3
\end{align*}

### Kinematics

\begin{align*}
\dot{\delta p_i} &= \delta v_i \\
\text{Continuous Case} \qquad \qquad \dot{\delta v_i} &= 0 \\
\dot{\delta \theta_i} &= 0 \\
\\
\delta p_i &\leftarrow \delta p_i + \delta v_i \Delta t + \delta p_{in} \\
\text{Discrete Case} \qquad \qquad \delta v_i &\leftarrow \delta v_i + \delta v_{in} \\
\delta \theta_i &\leftarrow \delta \theta_i + \delta \theta_{in}\\
\end{align*}

## Combined State

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b & \delta p_i & \delta v_i & \delta \theta_i & \delta p_{i + 1} & \delta v_{i + 1} & \delta \theta_{i + 1} & \cdots \end{bmatrix}^\top \\
\end{align*}

We need to define the error state kinematic functions for the jacobian matrix to propagate the error in the state error covariance matrix. We never actually propagate the error state with these functions but they are essential to cunstruct the matrix

$\delta v_{n}$, $\delta \theta_{n}$, $\delta a_n$ and $\delta \omega_n$ are the random impulses applied to the velocity, orientation and bias estimates, modeled by white Gaussian processes (See Solà). Their mean is zero, and their covariances matrices are obtained by integrating the covariances of $\tilde a_n$, $\tilde \omega_n$, $a_{bn}$ and $\omega_{bn}$ over the step time $\Delta t$.
This is actually never really used since we never really calculate the error state directly, its more like the covariance matrices of this value get added to the error state covariance matrix.

---
# Measurements

## IMU

The measurements are mainly used to correct the state in the update step of the filter. In the case of the IMU part of the state, the accelerometer can be used as both an input and a measurement to the filter. When the IMU is precieved to be stationary because there is (almost) no angular velocity coming from the gyroscope and (almost) no acceleration coming from the accelerometer (except for gravity), we can correct pitch and roll in our orientation.

The measured acceleration is then compared to what we would predict gravity is based on the predicted orientation:

\begin{align*}
h_a = \mathbf{R}^{\{q\}}g + a_b
\end{align*}

## Face detection

We can correct the speakers position and orientation from the output of the face detection algorithm. But since our position and orientation are represented globally in the state, we need to transform them into the camera space.

\begin{align*}
h_{sq} &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
h_{sp} &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\\
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

One important caviat is the fact that the IMU and the camera do not share the same coordinate system. The difference in rotation and translation is expressed as with the $\Delta$ vectors.

## Combined Measurements

\begin{align*}
h_a &= \mathbf{R}^{\{q\}}g + a_b \\
h_{sq} &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
h_{sp} &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\end{align*}

---
# Observability

## Weak Observability of Nominal State

Local weak Observability of non-linear systems (derived from Propositions 90 and 92, Adamy 2025):

A system in the form of

\begin{align*}
\dot x &= f(x,u) \\
y &= h(x)
\end{align*}

with $x$ defined over $D_{x, \text{def}} \subseteq \mathbb{R}^n$ is weakly observable if the following condition is met:

\begin{align*}
\text{rank}\left( \begin{bmatrix}
\frac{\partial L_f^0 h(x)}{\partial x} \\
\frac{\partial L_f^1 h(x)}{\partial x} \\
\vdots \\
\frac{\partial L_f^{n-1} h(x)}{\partial x} \\
\end{bmatrix} \right) = n
\end{align*}

### Practical Calculation

We can construct this matrix by taking the derivatives of our measurement function one after the other

\begin{align*}
y &= h(x) \\
\dot y &= \frac{d}{dt} y = \frac{d}{dt} h(x) = \frac{\partial h}{\partial x} \cdot \frac{dx}{dt} = \frac{\partial h}{\partial x} \cdot \dot x &&\Rightarrow \frac{\partial h}{\partial x} f(x,u) \\
\ddot y &= \frac{d}{dt} \dot{y} = \frac{\partial \dot{y}}{\partial x} \frac{dx}{dt} &&\Rightarrow \frac{\partial \dot{y}}{\partial x} f(x,u) \\
\dddot y &= \frac{d}{dt} \ddot{y} &&\Rightarrow \frac{\partial \ddot{y}}{\partial x} f(x,u) \\
& &&\vdots \\
y^{(k)} &= L_f^k h(x) &&\Rightarrow \frac{\partial y^{(k-1)}}{\partial x} \cdot f(x,u)
\end{align*}

$L_f^k h(x)$ stands for the Lie Derivative but follows the simple pattern as per the last row. Next up, we put the derivatives together in a vector and take the partial derivative with respect to $x$:

\begin{align*}
\frac{\partial}{\partial x} \begin{bmatrix} y \\ \dot{y} \\ \ddot{y} \\ \vdots \\ y^{(k)}\end{bmatrix} = \begin{bmatrix} \frac{\partial}{\partial x} h \\ \frac{\partial}{\partial x} \dot{y} \\ \frac{\partial}{\partial x} \ddot{y} \\ \vdots \\ \frac{\partial}{\partial x} y^{(k)}\end{bmatrix} \Rightarrow \begin{bmatrix} H_0 \\ H_1 \\ H_2 \\ \vdots \\ H_k \end{bmatrix} = \begin{bmatrix}
\frac{\partial h}{\partial x} \\
\frac{\partial}{\partial x} \left(H_0 \cdot f(x,u)\right) \\
\frac{\partial}{\partial x} \left(H_1 \cdot f(x,u)\right) \\
\frac{\partial}{\partial x} \left(H_2 \cdot f(x,u)\right) \\
\vdots \\
\frac{\partial}{\partial x} \left(H_{k-1} \cdot f(x,u)\right) \\
\end{bmatrix} \Rightarrow 
\begin{bmatrix}
\frac{\partial L_f^0 h(x)}{\partial x} \\
\frac{\partial L_f^1 h(x)}{\partial x} \\
\frac{\partial L_f^2 h(x)}{\partial x} \\
\vdots \\
\frac{\partial L_f^{k} h(x)}{\partial x} \\
\end{bmatrix}
\end{align*}

Now, instead of doing this up until $k$ vector entries, we do it until $n$ entries according to the definition of $x$ with $D_{x, \text{def}} \subseteq \mathbb{R}^n$. In the case of this observer with a state consisting of $x_\text{IMU} \in SO(3) \times \left(\mathbb{R}^3\right)^4$ and $x_s \in \prod_{i = 1}^n \left(SO(3) \times \left(\mathbb{R}^3\right)^2 \right)$, $n$ is equal to $26$.

*Notes* by Gemini:

- the nominal Jacobian matrix will max out at rank $n_{DOF}$ (e.g., 24), not 26, unless you explicitly enforce the norm constraint $\dot{q} = \frac{1}{2} \Omega q$ or evaluate using 3D minimal parameterizations.
- You do not necessarily need $n$ Lie derivative orders to reach rank $n$; you often reach full rank in far fewer than $n-1$ derivatives because each order yields $\text{dim}(y)$ scalar equations.

# Jacobians for the Filter Equations